In [ ]:
from anndata import read_h5ad

adata = read_h5ad("./data/larry/postprocessed.h5ad")
adata

In [ ]:
# --- Block 1: Force scVelo to plot precomputed velocity embeddings ------------
import scvelo as scv
from anndata import read_h5ad
import numpy as np

scv.settings.verbosity = 3
scv.settings.set_figure_params('scvelo', dpi=120, figsize=(6, 5))

# Load data
adata = read_h5ad("./data/larry/postprocessed.h5ad")

print("Available embeddings:", list(adata.obsm.keys()))
print("Available velocity embeddings:", [k for k in adata.obsm.keys() if 'velocity' in k])

# ------------------------------------------------------------------------------
# 💥 Rename your embeddings to match what scVelo expects ("umap" convention)
# ------------------------------------------------------------------------------

# Backup originals (optional)
adata.obsm["X_emb_original"] = adata.obsm["X_emb"].copy()

# Force rename so scVelo is happy
adata.obsm["X_umap"] = adata.obsm["X_emb"].copy()

# 1️⃣ Plot with velocity_emb
if "velocity_emb" in adata.obsm:
    adata.obsm["velocity_umap"] = adata.obsm["velocity_emb"].copy()
    print("✅ Plotting precomputed velocity_emb as velocity_umap ...")
    scv.pl.velocity_embedding_stream(
        adata,
        basis="umap",
        vkey="velocity",
        color="state_info",
        title="Velocity streamlines (velocity_emb)",
        density=1.5,
        smooth=0.8,
        alpha=0.8,
    )
else:
    print("⚠️ No velocity_emb found in adata.obsm.")

# 2️⃣ Plot with velocity_pyro_emb
if "velocity_pyro_emb" in adata.obsm:
    adata.obsm["velocity_umap"] = adata.obsm["velocity_pyro_emb"].copy()
    print("✅ Plotting precomputed velocity_pyro_emb as velocity_umap ...")
    scv.pl.velocity_embedding_stream(
        adata,
        basis="umap",
        vkey="velocity",
        color="state_info",
        title="Velocity streamlines (velocity_pyro_emb)",
        density=1.5,
        smooth=0.8,
        alpha=0.8,
    )
else:
    print("⚠️ No velocity_pyro_emb found in adata.obsm.")

In [ ]:
# --- Block 2: Compare all velocity matrices on UMAP --------------------------
import scvelo as scv
from anndata import read_h5ad
import matplotlib.pyplot as plt

scv.settings.verbosity = 3
scv.settings.set_figure_params('scvelo', dpi=120, figsize=(6, 5))

# Load data
adata = read_h5ad("./data/larry/postprocessed.h5ad")

# ------------------------------------------------------------------------------
# 1️⃣ Check available velocity layers
# ------------------------------------------------------------------------------
velocity_layers = [k for k in adata.layers.keys() if "velocity" in k]
print("Available velocity layers:", velocity_layers)

# ------------------------------------------------------------------------------
# 2️⃣ Ensure we have a neighbor graph and UMAP embedding
# ------------------------------------------------------------------------------
if "neighbors" not in adata.uns:
    print("Computing neighborhood graph...")
    scv.pp.neighbors(adata, n_neighbors=30, n_pcs=30)

if "X_umap" not in adata.obsm:
    print("Computing UMAP embedding...")
    scv.tl.umap(adata)

# ------------------------------------------------------------------------------
# 3️⃣ Plot each velocity field on the same UMAP embedding
# ------------------------------------------------------------------------------
for vkey in ["velocity", "velocity_pyro"]:
    if vkey in adata.layers:
        print(f"\n🚀 Plotting {vkey} on UMAP ...")
        scv.pl.velocity_embedding_stream(
            adata,
            basis="umap",
            vkey=vkey,
            color="state_info",       # cell state or cluster info
            title=f"RNA velocity on UMAP ({vkey})",
            density=1.5,
            smooth=0.8,
            alpha=0.8,
            legend_loc="none",
        )
    else:
        print(f"⚠️ Layer '{vkey}' not found in adata.layers.")


In [ ]:
# --- Block 3 (Static Velocity Inference; scVelo ≥0.3.x compatible) ---
import scvelo as scv
from anndata import read_h5ad

scv.settings.verbosity = 3
scv.settings.set_figure_params("scvelo", dpi=120, figsize=(6, 5))

# Load dataset
adata = read_h5ad("./data/larry/postprocessed.h5ad")

# ------------------------------------------------------------------------------
# 1️⃣ Preprocessing: filtering, normalization, moments
# ------------------------------------------------------------------------------
scv.pp.filter_and_normalize(
    adata,
    min_counts=20,
    min_cells=3,
    n_top_genes=2000,
    flavor="seurat",
    subset_highly_variable=True,
    log=True,
    enforce=True
)

scv.pp.moments(
    adata,
    n_pcs=30,
    n_neighbors=30
)

# ------------------------------------------------------------------------------
# 2️⃣ Compute velocities (stochastic / steady-state model)
# ------------------------------------------------------------------------------
scv.tl.velocity(
    adata,
    mode="stochastic",   # ✅ static inference, no full kinetic fitting
    vkey="velocity",
    use_raw=False
)

# ------------------------------------------------------------------------------
# 3️⃣ Build velocity graph and embedding
# ------------------------------------------------------------------------------
scv.tl.velocity_graph(
    adata,
    vkey="velocity",
    n_neighbors=30,
    approx=False,
    n_jobs=8
)

# Compute or reuse UMAP
if "X_umap" not in adata.obsm:
    scv.tl.umap(
        adata,
        min_dist=0.4,
        spread=1.0,
        random_state=0
    )

# ------------------------------------------------------------------------------
# 4️⃣ Visualization: velocity streamlines on UMAP
# ------------------------------------------------------------------------------
scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    vkey="velocity",
    color="state_info",
    title="RNA Velocity (Stochastic Model)",
    density=1.5,
    smooth=0.8,
    alpha=0.8,
    legend_loc="none"
)

In [ ]:
import pandas as pd

# Suppose your time_info values are numeric (e.g. 0, 24, 48)
print("Unique time points before conversion:", adata.obs["time_info"].unique())

# Convert to string and categorical dtype
adata.obs["time_info_cat"] = adata.obs["time_info"].astype(str)
adata.obs["time_info_cat"] = pd.Categorical(adata.obs["time_info_cat"], 
                                            categories=sorted(adata.obs["time_info_cat"].unique()),
                                            ordered=True)

print("Converted categories:", adata.obs["time_info_cat"].cat.categories)


scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="time_info_cat",       # use the categorical column!
    title="RNA velocity by time point",
    legend_loc="right margin",   # optional: to show the legend
    density=1.5,
    smooth=0.8,
    alpha=0.3,
    arrow_size=2.0
)

In [ ]:
import matplotlib.pyplot as plt
import scvelo as scv

# Ensure we have categorical labels
if "time_info_cat" not in adata.obs:
    adata.obs["time_info_cat"] = adata.obs["time_info"].astype(str)

time_points = sorted(adata.obs["time_info_cat"].unique())

# Get the 2D embedding
X_umap = adata.obsm["X_umap"]

# Make one subplot per time point
fig, axes = plt.subplots(1, len(time_points), figsize=(12, 4), tight_layout=True)

for ax, t in zip(axes, time_points):
    highlight = adata.obs["time_info_cat"] == t

    # First, draw grey background (all other cells)
    ax.scatter(
        X_umap[~highlight, 0], X_umap[~highlight, 1],
        c="lightgrey", s=8, alpha=0.4, linewidths=0
    )

    # Then, overlay red for the selected time point
    ax.scatter(
        X_umap[highlight, 0], X_umap[highlight, 1],
        c="red", s=12, alpha=0.9, linewidths=0
    )

    ax.set_title(f"Time {t}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_frame_on(False)

plt.show()


In [ ]:
import anndata as ad

# Keep all cell metadata
obs_keep = adata.obs.copy()

# Construct minimal layers dictionary
layers_keep = {
    "Ms": adata.layers.get("Ms"),
    "Mu": adata.layers.get("Mu"),
    "velocity": adata.layers.get("velocity"),
}

# Include latent time if present in layers
if "fit_t" in adata.layers:
    layers_keep["fit_t"] = adata.layers["fit_t"]
    print("✅ Included 'fit_t' from layers.")
else:
    print("⚠️ 'fit_t' not found in layers; skipping.")

# Build a minimal AnnData
adata_min = ad.AnnData(
    X=adata.X,                # normalized expression matrix
    obs=obs_keep,             # metadata
    var=adata.var.copy(),     # gene info
    obsm=adata.obsm.copy(),   # embeddings (e.g. X_umap, velocity_umap)
    layers=layers_keep,       # Ms, Mu, velocity, fit_t
    uns={"velocity_params": adata.uns.get("velocity_params", {})}
)

# Optionally: restrict to HVGs to reduce file size
if "highly_variable" in adata.var.columns:
    hvgs = adata.var.index[adata.var["highly_variable"]]
    adata_min = adata_min[:, hvgs].copy()
    print(f"🧬 Retained {len(hvgs)} HVGs in minimal object.")

# Save
adata_min.write_h5ad("./data/larry/adata_minimal_velocity.h5ad", compression="gzip")
print("💾 Saved minimal velocity object → ./data/larry/adata_minimal_velocity.h5ad")